# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL as defined below.

In [ ]:
# Install mlcroissant library (if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We inspect the dataset record sets and their fields using their Croissant `@id` attributes.

In [ ]:
# List available record sets with their @id and field @id's and names
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f'Record set name:      {rs.name}')
        print(f'  @id:                {rs.id}')
        print(f'  Description:        {getattr(rs, "description", "")}')
        print('  Fields:')
        for f in rs.fields:
            print(f'    - {f.name} (@id: {f.id}) [type: {f.data_type}]')
        print('-' * 48)
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for further analysis.

All record sets and fields are referenced by their Croissant `@id`.

In [ ]:
# Gather all record set @ids
record_sets = []
record_set_names = {}
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_sets.append(rs.id)
        record_set_names[rs.id] = rs.name

# Load each record set into a DataFrame
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set '{record_set_names[rs_id]}' (@id: {rs_id}) with {len(df)} records. Columns:")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print(f"Record set '{record_set_names[rs_id]}' (@id: {rs_id}) contains 0 records.")

# For demonstration, choose the first non-empty record set
main_record_set_id = None
for k, v in dataframes.items():
    if v.shape[0] > 0:
        main_record_set_id = k
        break
if main_record_set_id:
    print(f"\nMain record set selected: {main_record_set_id} ({record_set_names[main_record_set_id]})")

## 4. Exploratory Data Analysis (EDA)

Here, we select a numeric field and perform basic analysis such as filtering, normalization, and grouping. Field references use their Croissant `@id` attributes.

In [ ]:
# Identify numeric fields in the main record set
import numpy as np
numeric_field_id = None
group_field_id = None

if main_record_set_id:
    recset = None
    for rs in metadata.record_sets:
        if rs.id == main_record_set_id:
            recset = rs
            break
    num_fields = [f for f in recset.fields if f.data_type in ('Float', 'Integer', 'Number')]
    cat_fields = [f for f in recset.fields if f.data_type == 'Text']
    # Pick the first available numeric field
    if num_fields:
        numeric_field_id = num_fields[0].id
        print(f"Using numeric field: {num_fields[0].name} (@id: {numeric_field_id})")
    if cat_fields:
        group_field_id = cat_fields[0].id
        print(f"Using group field: {cat_fields[0].name} (@id: {group_field_id})")
else:
    print('No data available for EDA.')

# Continue analysis if numeric field is available
if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]

    # Clean data (convert to numeric)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() # Use mean as example threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize the field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional: group by a text column
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show the histogram of the selected numeric field and, if available, a grouped bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for the numeric field
if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

# If grouped data available, show bar plot
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we:
1. Loaded and examined metadata, record sets, and fields using the Croissant `@id` convention.
2. Loaded data from the available record sets into pandas DataFrames.
3. Performed exploratory analysis focused on numeric and categorical fields.
4. Visualized data distributions to support downstream bioinformatics or proteomics research.

Refer to the [FAIR² repo](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for full dataset documentation and proper citation.